# 06 · Decisión y acción clínica — Etapas ⑤ y ⑥ del pipeline

**Componentes del pipeline:** Decision Making Module (umbrales de riesgo) + Clinical Action Component.

**Entrada:** `resultados/scores_test.csv` (probabilidades calibradas de `05`) + `data/raw/diabetic_data.csv` (solo `race`, para equidad)
**Salida:** `resultados/06_costos_politicas.csv`, `resultados/06_equidad.csv`, `resultados/06_DCA.png`

| Riesgo | Umbral | Acción clínica |
|--------|--------|----------------|
| Bajo | `p < t1` | Alta rutinaria y seguimiento estándar |
| Moderado | `t1 ≤ p < t2` | Monitoreo reforzado, consulta farmacéutica |
| Alto | `p ≥ t2` (con cupo K) | Plan de transición personalizado, salud domiciliaria |

> Umbrales y costos derivan de la matriz de costos (mismos parámetros que `07`). Aquí se aplican sobre el conjunto de **test** con las probabilidades calibradas de `05`, como política de despliegue.

In [ ]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import logit, expit
from scipy.optimize import minimize_scalar

import sklearn
from sklearn.linear_model import LogisticRegression

from src.paths import RAW, PROCESSED, RESULTADOS

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
SEED = 42
MODO_RAPIDO = False
B_BOOT = 100 if MODO_RAPIDO else 500
_SK18 = tuple(int(x) for x in sklearn.__version__.split(".")[:2]) >= (1, 8)

## 1. Cargar probabilidades calibradas (05) y `race` (crudo, solo equidad)

In [ ]:
scores = pd.read_csv(RESULTADOS / "scores_test.csv")
raw_race = pd.read_csv(RAW / "diabetic_data.csv", na_values=["?"], keep_default_na=False,
                       low_memory=False)[["encounter_id", "race"]]
d = scores.merge(raw_race, on="encounter_id", how="left")
d["race"] = d["race"].fillna("Desconocido")
d["edad"] = d["age_num"]
d = d.rename(columns={"y_true": "y", "p_cal": "p", "p_lo": "lo", "p_hi": "hi"})
print("Test:", d.shape, "| prevalencia <30:", round(d.y.mean(), 4))

## 2. Parámetros de decisión (matriz de costos)

In [ ]:
# ---------------- Parámetros de decisión (matriz de costos) ----------------
C_R   = 16037.0   # costo de un reingreso (US$)            [Evidencia: Ghabowen et al., 2024]
C_I   = 550.0     # costo intervención Alto Riesgo (US$)   [Supuesto]
E     = 0.18      # eficacia intervención Alto (RR 0.82)   [Evidencia: Leppin et al., 2014]
C_M   = 50.0      # costo intervención Moderado (US$)      [Supuesto]
E_M   = 0.03      # eficacia intervención Moderado         [Supuesto]
E_SENSIBILIDAD = [0.12, 0.18, 0.38]
K_ALTO = 0.15     # máximo de altas marcadas Alto Riesgo [Supuesto]
FACTOR_OCUPACION = 1.25  # ocupación > 85%: costo efectivo del reingreso +25%

def p_estrella(c_i=C_I, c_r=C_R, e=E):
    "Umbral óptimo intervenir vs. no intervenir: p* = C_i / (C_r · e)"
    return c_i / (c_r * e)

def umbrales(c_r=C_R, c_i=C_I, e=E, c_m=C_M, e_m=E_M):
    "t1: Bajo->Moderado ; t2: Moderado->Alto (de la matriz de costos)"
    t1 = c_m / (c_r * e_m)
    t2 = (c_i - c_m) / (c_r * (e - e_m))
    return t1, t2

T1, T2 = umbrales()
tabla_param = pd.DataFrame([{
    "e": e, "p*": p_estrella(e=e), "peso FN:FP equiv.": (1 - p_estrella(e=e)) / p_estrella(e=e),
    "t1": umbrales(e=e)[0], "t2": umbrales(e=e)[1]} for e in E_SENSIBILIDAD]).round(3)
print(f"Caso base -> p* = {p_estrella():.3f} | t1 = {T1:.3f} | t2 = {T2:.3f}")
tabla_param

## 3. Utilidades de decisión, costos y beneficio neto

In [ ]:
def asignar(p, lo, hi, t1, t2, k=K_ALTO):
    n = len(p); cat = np.full(n, "Bajo", dtype=object)
    cat[p >= t1] = "Moderado"
    cand = np.where(p >= t2)[0]
    orden = cand[np.argsort(-p[cand])]
    cupo = int(np.floor(k * n))
    cat[orden[:cupo]] = "Alto"
    prioridad = np.zeros(n, bool); prioridad[orden[cupo:]] = True
    incierto = ((lo < t1) & (hi >= t1)) | ((lo < t2) & (hi >= t2))
    return cat, prioridad, incierto

def costo_categorias(y, cat, e=E, e_m=E_M, c_r=C_R):
    y = np.asarray(y)
    c = np.where(cat == "Alto", C_I + c_r * y * (1 - e),
                 np.where(cat == "Moderado", C_M + c_r * y * (1 - e_m), c_r * y))
    return 1000 * c.mean()

def costo_por_1000(y, intervenir, e=E, c_i=C_I, c_r=C_R):
    y = np.asarray(y); m = np.asarray(intervenir).astype(bool)
    vp = np.sum(m & (y == 1)); fp = np.sum(m & (y == 0)); fn = np.sum(~m & (y == 1))
    return 1000 * (c_i * (vp + fp) + c_r * (fn + vp * (1 - e))) / len(y)

def beneficio_neto(y, p, pt):
    y = np.asarray(y); marca = np.asarray(p) >= pt; n = len(y)
    vp = np.sum(marca & (y == 1)); fp = np.sum(marca & (y == 0))
    return vp / n - fp / n * pt / (1 - pt)

def logistica_sin_penalizacion():
    return LogisticRegression(C=np.inf, max_iter=1000) if _SK18 else LogisticRegression(penalty=None, max_iter=1000)

def pendiente_intercepto(y, p):
    lp = logit(np.clip(p, 1e-6, 1 - 1e-6))
    pendiente = logistica_sin_penalizacion().fit(lp.reshape(-1, 1), y).coef_[0, 0]
    def nll(a):
        q = np.clip(expit(a + lp), 1e-9, 1 - 1e-9)
        return -np.sum(y * np.log(q) + (1 - y) * np.log(1 - q))
    intercepto = minimize_scalar(nll, bounds=(-5, 5), method="bounded").x
    return pendiente, intercepto

## 4. Clasificación de riesgo y acción clínica

In [ ]:
t1, t2 = umbrales()
cat, prioridad, incierto = asignar(d.p.values, d.lo.values, d.hi.values, t1, t2)
d["cat"] = cat; d["prioridad"] = prioridad; d["incierto"] = incierto
ACCIONES = {"Bajo": "Alta rutinaria y seguimiento estándar",
            "Moderado": "Monitoreo reforzado, consulta farmacéutica",
            "Alto": "Plan de transición personalizado, salud domiciliaria"}
d["accion"] = d["cat"].map(ACCIONES)

orden = ["Bajo", "Moderado", "Alto"]
resumen = pd.DataFrame({
    "% pacientes": d.cat.value_counts(normalize=True).reindex(orden),
    "tasa real de reingreso": d.groupby("cat").y.mean().reindex(orden),
    "acción clínica": pd.Series(ACCIONES).reindex(orden),
})
recall_alto = ((d.cat == "Alto") & (d.y == 1)).sum() / d.y.sum()
display(resumen)
print(f"Recall de Alto Riesgo: {recall_alto:.1%} -> impacto proyectado = e × recall = {E * recall_alto:.1%}")
print(f"Tasa de Alto: {(d.cat=='Alto').mean():.1%} (cupo {K_ALTO:.0%}) | prioridad desbordada: {d.prioridad.mean():.1%} | "
      f"% incertidumbre alta: {d.incierto.mean():.1%}")

## 5. Costo esperado por política (por 1,000 altas)

In [ ]:
def tabla_costos(e=E, c_r=C_R, etiqueta=""):
    t1, t2 = umbrales(c_r=c_r, e=e)
    cat_e, _, _ = asignar(d.p.values, d.lo.values, d.hi.values, t1, t2)
    y = d.y.values
    r = {"Modelo (3 categorías)": costo_categorias(y, cat_e, e=e, c_r=c_r),
         "No intervenir": 1000 * c_r * y.mean(),
         "Intervenir a todos": costo_categorias(y, np.full(len(y), "Alto", dtype=object), e=e, c_r=c_r),
         "Modelo binario (p ≥ p*)": costo_por_1000(y, d.p.values >= p_estrella(e=e, c_r=c_r), e=e, c_r=c_r)}
    r["Ahorro vs mejor referencia"] = min(r["No intervenir"], r["Intervenir a todos"]) - r["Modelo (3 categorías)"]
    return pd.Series(r, name=f"{etiqueta} (t1={t1:.2f}, t2={t2:.2f})")

tabla_costos_final = pd.DataFrame(
    [tabla_costos(etiqueta="Base e=0.18")] +
    [tabla_costos(e=e, etiqueta=f"e={e}") for e in E_SENSIBILIDAD] +
    [tabla_costos(c_r=C_R * FACTOR_OCUPACION, etiqueta="Ocupación >85%")])
tabla_costos_final.to_csv(RESULTADOS / "06_costos_politicas.csv")
print("Costo esperado por 1,000 altas (US$):")
tabla_costos_final.round(0)

## 6. Análisis de curva de decisión (DCA)

In [ ]:
pts = np.round(np.arange(0.01, 0.41, 0.01), 2)
prev = d.y.mean()
nb_mod = np.array([beneficio_neto(d.y, d.p, pt) for pt in pts])
nb_todos = prev - (1 - prev) * pts / (1 - pts)
rango = (pts >= 0.05) & (pts <= 0.30)
mejor_ref = np.maximum(nb_todos, 0)
dca_ok = bool(np.all(nb_mod[rango] >= mejor_ref[rango] - 1e-9) and np.any(nb_mod[rango] > mejor_ref[rango]))
frac_supera = np.mean(nb_mod[rango] > mejor_ref[rango] + 1e-9)

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(pts, nb_mod, lw=2, label="Modelo calibrado (test)")
ax.plot(pts, nb_todos, "--", label="Intervenir a todos")
ax.axhline(0, c="k", lw=1, label="No intervenir")
ax.axvspan(0.05, 0.30, color="grey", alpha=0.1, label="Rango evaluado")
for e in E_SENSIBILIDAD:
    ax.axvline(p_estrella(e=e), c="r", ls=":", lw=1)
ax.set_ylim(-0.02, max(0.05, nb_mod.max() * 1.3))
ax.set_xlabel("Umbral de probabilidad"); ax.set_ylabel("Beneficio neto")
ax.set_title("Figura 06.1 — Curva de decisión (líneas rojas: p* para e = 0.12 / 0.18 / 0.38)")
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(RESULTADOS / "06_DCA.png", bbox_inches="tight"); plt.show()
print(f"DCA: el modelo supera a ambas referencias en {frac_supera:.0%} del rango 0.05–0.30 | criterio cumplido: {dca_ok}")

## 7. Auditoría de equidad

In [ ]:
MIN_POS = 100
d["grupo_edad"] = np.where(d.edad >= 70, "≥70", "<70")

def fnr(s):
    return np.mean(s.cat.values[s.y.values == 1] != "Alto")

def auditar(col, ref):
    filas = []; base = d[d[col] == ref]
    for g, s in d.groupby(col):
        n_pos = int(s.y.sum())
        fila = {"dimensión": col, "grupo": g, "n": len(s), "reingresos": n_pos,
                "evaluable": n_pos >= MIN_POS, "tasa_reingreso": s.y.mean(),
                "tasa_selección_Alto": (s.cat == "Alto").mean(), "FNR": fnr(s)}
        if n_pos >= MIN_POS:
            fila["razón_FNR"] = fnr(s) / fnr(base)
            r = np.random.default_rng(SEED); vals = []
            ps, pb = s[s.y == 1], base[base.y == 1]
            for _ in range(B_BOOT):
                a = ps.iloc[r.integers(0, len(ps), len(ps))]; c = pb.iloc[r.integers(0, len(pb), len(pb))]
                vals.append(fnr(a) / fnr(c))
            fila["razón_FNR_ic_inf"], fila["razón_FNR_ic_sup"] = np.percentile(vals, [2.5, 97.5])
            fila["pendiente_cal"], fila["intercepto_cal"] = pendiente_intercepto(s.y.values, s.p.values)
        filas.append(fila)
    return pd.DataFrame(filas)

equidad = pd.concat([auditar("grupo_edad", "<70"), auditar("race", "Caucasian")], ignore_index=True)
equidad["cumple_FNR"] = equidad.razón_FNR.between(0.87, 1.15)
equidad["cumple_calibración"] = equidad.pendiente_cal.between(0.80, 1.20) & (equidad.intercepto_cal.abs() <= 0.20)
equidad.to_csv(RESULTADOS / "06_equidad.csv", index=False)
display(equidad.round(3))
print("Grupos excluidos por < 100 reingresos:", equidad.loc[~equidad.evaluable, "grupo"].tolist())